##### **Linear Regression using Stochastic Gradient Descent**

##### Dataset  https://www.kaggle.com/competitions/boston-housing/data

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
data_overview={
"crim":  "per capita crime rate by town.",
"zn" : "proportion of residential land zoned for lots over 25,000 sq.ft.",
"indus" : "proportion of non-retail business acres per town.",
"chas" : "Charles River dummy variable (= 1 if tract bounds river; 0 otherwise).",
"nox" : "nitrogen oxides concentration (parts per 10 million).",
"rm" : "average number of rooms per dwelling.",
"age" : "proportion of owner-occupied units built prior to 1940.",
"dis" : "weighted mean of distances to five Boston employment centres.",
"rad" : "index of accessibility to radial highways.",
"tax" : "full-value property-tax rate per $10,000.",
"ptratio" : "pupil-teacher ratio by town.",
"black" : "1000(Bk - 0.63)^2 where Bk is the proportion of blacks by town.",
"lstat": "lower status of the population (percent).",
"medv" : "median value of owner-occupied homes in $1000s"
}

def tell_me_about(field):
    return data_overview[field]

In [ ]:
tell_me_about("medv")

In [ ]:
train_df=pd.read_csv('data/train.csv')
train_df.head()

In [ ]:
train_df.columns

In [ ]:
# fe_columns=[ 'crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'black', 'lstat', 'medv']
# ID is usually not needed so we ignore that

## Part 1 (Basic EDA)

In [ ]:
train_df.describe()

In [ ]:
train_df.info()
## everything is numeric and non null

In [ ]:
train_df.shape

In [ ]:
def basic_summary(df):
    summary_info ={
        "missing_values" : df.isnull().sum(),
        "dtypes" : df.dtypes, 
        "stats" : df.describe().T
    }
    print(summary_info)

In [ ]:
basic_summary(train_df)

In [ ]:
plt.figure(figsize=(12,5))
sns.histplot(train_df['medv'], kde=True, bins=30)
plt.title('Distribution of target (medv)')
plt.xlabel('Median value of homes ($1000s)')
plt.ylabel('Count')
plt.show()

## This is almost normalLY distributed 

In [ ]:
plt.figure(figsize=(12,8))
corr = train_df.drop(columns='ID').corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation heatmap')
plt.show()

## Most of the columns are highly correlated with other variables 

In [ ]:
# plt.figure(figsize=(15,10))
train_df.drop(columns='ID').hist(bins=30)
plt.suptitle('Feature Distributions', fontsize=16)
plt.show()

In [ ]:
numeric_cols=train_df.drop(columns=['ID', 'chas', 'medv']).columns

# chas is categorical column having only 0, 1 
# ID doesn't influence anything
# medv is the output/target variable 

In [ ]:
tell_me_about("chas")

In [ ]:
plt.figure(figsize=(15,10))
for i, col in enumerate(numeric_cols,1):
    plt.subplot(4,3,i)
    sns.boxplot(x=train_df[col])
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
## Since the columns are having different numeric ranges, we should do the scaling too

## Part 2 (Model Development)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
## Features and target
X_train_full = train_df.drop(columns= ['ID', 'medv'])
y_train_full = train_df['medv']

In [ ]:
X_train_split, X_test_split, y_train_split, y_test_split = train_test_split (
    X_train_full, y_train_full, test_size=0.20, random_state=42) 
## test_size of 0.20 suggests 80 20 split i.e. you are using 80% of the dfata for training and will test on remaining 20 %

In [ ]:
train_df.columns

In [ ]:
numeric_features = ['crim', 'zn', 'indus', 'nox', 'rm', 'age', 'dis', 'tax', 'ptratio', 'black', 'lstat']
categorical_features = ['chas', 'rad' ]

In [ ]:
tell_me_about("rad")

In [ ]:
numeric_transformer = StandardScaler()

In [ ]:
categorical_transformer = OneHotEncoder(handle_unknown='ignore', drop  ='first') 
## handle_unknown is important to be ignored, because u might have instances where the type of data say a particular color was absent in the training dataset 
# but might come up in test dataset, so we need to ignore that

In [ ]:
# Create the preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

**Difference between fit_transform and transform**

- fit_transform() : Learns parameters from the data (fit) AND applies the transformation. Used on **training data only**. 
    - fit_transform(X_train) learns: mean of each feature, standard deviation and then scales the training data.
    -  fit_transform() = learn + apply,
- transform() : Applies an already-learned transformation. Used on **test or new unseen data**
    - transform(X_test) uses the mean and std from the training data, not from the test data.
    - transform = apply only (using previously learned state)


**Why not use fit_transform on test data?**

Because that would leak information from the test set into the training process (data leakage). The model must not learn anything from the test data.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train_split)
X_test_processed = preprocessor.transform(X_test_split)
# 01:07:51

In [ ]:
print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape: {X_test_processed.shape}")

In [ ]:
# TRAIN MODEL USING SGDREGRESSOR
sgd_reg = SGDRegressor(
    max_iter=1000,
    eta0=0.01, # Learning rate
    random_state=42
)

# IN ML ALGO -  we only have fit & predict
## FIT_TRANSOFRM, FIT, TRANSFORM ARE THERE FOPR SCALING, NORMALISATION ETC. 

In Linear Regression, learning can be done in 2 ways: 
- **Gradient Descent** : Here you update the parameters iteratively i.e. there is a lot of computation involved. So it is suitable for medium and large data set so that computation costs make sense. Not ideal for small datasets. This is present in sklearn under SGD Regressor
- **OLS** : This is good for small to medium sized datasets. Here it finds the optimal parameters in a single pass. This is faster and much easier hence it becomes the default model for testing. 

In [ ]:
sgd_reg.fit(X_train_processed, y_train_split)

In [ ]:
# 20 features --> y = MX + C 
# M --> 20m
# C --> 1

sgd_reg.coef_, sgd_reg.intercept_

In [ ]:
len(sgd_reg.coef_)

In [ ]:
# Here we are trying to run using 2 models SGD, OLS. The basic workflow is teh same. 

# Test Model

In [ ]:
# With SGD
y_test_pred = sgd_reg.predict(X_test_processed)
y_train_pred = sgd_reg.predict(X_train_processed)

In [ ]:
def plot_actual_vs_predicted_lines(y_actual, y_predicted):
    plt.figure(figsize=(14, 7))

    if isinstance(y_actual, pd.Series):
        y_actual = y_actual.values
    
    plt.plot(y_actual, label='Actual Values', color='#1f77b4', linewidth=2)

    plt.plot(y_predicted, label='Predicted Values', color='#ff7f0e', linewidth=2, linestyle='--')
    
    plt.title('Actual vs. Predicted Values Over Index (Trend Comparison)')
    plt.xlabel('Data Index')
    plt.ylabel('Median Home Value')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_actual_vs_predicted_lines(y_test_split, y_test_pred)

In [ ]:
# Evaluation
from sklearn.metrics import mean_squared_error

In [ ]:
mse_train = mean_squared_error(y_train_split, y_train_pred)
mse_test = mean_squared_error(y_test_split, y_test_pred)

print(f"MSE (TRAIN) (SGD) : {mse_train}")
print(f"MSE (TEST) (SGD): {mse_test}")

# R-squared : A metric of Model Performance

- R-squared tells us how much of the variation in the target variable is explained by the model. Higher is better, but context matters. 
- R-squared is usually is a decimal number refering to a percentage i.e. R-squared of 0.80 means 80 % variance is explained by the model. 
- 0.80 is better than 0.60 R-squared i.e. 0.80 model is better. 

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
r2_sdg = r2_score(y_test_split, y_test_pred)


print(f"R2 Square (SGD) : {r2_sdg:.2f}")
print(f"R2 Square (SGD) : {r2_sdg}")


## Save Artefacts

you need three components: scaling, encoding, model

In [ ]:
import joblib

In [ ]:
model_filename = '2.31) Linear_regression_model_SGD_artefacts.joblib'

preprocessor_filename = '2.32) Linear_regression_model_SGD_data_processor_artefacts.joblib'

In [ ]:
joblib.dump(sgd_reg, model_filename)
joblib.dump(preprocessor, preprocessor_filename)

In [ ]:
print(f"\nModel saved to: {model_filename}")
print(f"Preprocessor saved to: {preprocessor_filename}")